In [50]:
import pandas as pd
import yfinance as yf
import seaborn as sns
import os
sns.set_theme(style="darkgrid")

In [51]:
list = pd.read_excel('c:\\Users\\tacon\\tacona1016\\input\\stock_list.xlsx', engine='openpyxl')
df = yf.download(list.ticker.to_list(), period = '10y', auto_adjust=True)['Close'].reset_index()

[*********************100%***********************]  20 of 20 completed


In [52]:
df = df.rename(columns=dict(zip(list['ticker'], list['name'])))

In [53]:
df = pd.melt(df, id_vars=['Date']).dropna()

In [60]:
df['Date'] = pd.to_datetime(df['Date']).dt.date

In [62]:
from sqlalchemy import create_engine, text

# DB 파일 생성 및 연결
engine = create_engine("sqlite:///mydb.sqlite")

In [ ]:
'''
# 테이블 생성
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS stock (
            Date DATE,
            Ticker VARCHAR(50),
            value DOUBLE,
            PRIMARY KEY (Date, Ticker)
        )
    """))
'''

In [64]:
import pandas as pd

# SQLite에 append
with engine.connect() as conn:
    df.to_sql("stock", con=engine, if_exists="append", index=False)


In [65]:
query_text = """select * from stock where 1=1 and
                Ticker in ('Gold', 'Silver')
                """
query = text(query_text)
with engine.connect() as conn:
    df = pd.read_sql(query, con=engine)

In [67]:
df = pd.pivot_table(df, index='Date', columns='Ticker', values='value', observed=True).reset_index()

In [69]:
df['Gold/Silver'] = df['Gold']/df['Silver']

In [71]:
import streamlit as st
# Streamlit 차트
st.title("📈 금/은 비율 추이")
st.line_chart(df.set_index("Date")["Gold/Silver"])

2025-07-29 22:34:40.531 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-29 22:34:40.961 
  command:

    streamlit run C:\Users\tacon\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-07-29 22:34:40.962 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-29 22:34:40.963 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-29 22:34:42.311 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-29 22:34:42.312 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-07-29 22:34:42.313 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()